# IB Integration

In [9]:
import tkinter as tk
from tkinter import ttk, messagebox
import threading
import time
from datetime import datetime
from collections import deque
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.patches import Rectangle
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
import warnings
from services.vol_service import VolService

ModuleNotFoundError: No module named 'services'

In [ ]:
class IBApp(EWrapper, EClient):
    def __init__(self, callback = None):
        EClient.__init__(self, self)
        self.callback = callback
        self.connected = False
        self.last_price = None
        self.bid_price = None
        self.ask_price = None
        self.historical_data = {}
        self.histDone = threading.Event()

    def error(self, reqId, errorCode, errorString, advancedOrderRejectJson= ""):
        if errorCode in [2104, 2106, 2158, 2176]:
            return
        if errorCode == 10167:
            print("Note: Delayed market data.")
            return
        print(f"Error | reqID: {reqId} | errorCode: {errorCode}  \n Msg: {errorString}")

    def nextValidId(self, orderId):
        self.connected = True
        print(f"Connected.")
    

    def historicalData(self, reqId, bar):
        if reqId not in self.historical_data:
            self.historical_data[reqId] = []
        self.historical_data[reqId].append({
            "o": bar.open,
            "h": bar.high,
            "l": bar.low,
            "c": bar.close,            
        })

    def historicalDataEnd(self, reqId, start, end):
        self.histDone.set()

    def tickPrice(self, reqId, tickType, price, attrib):
        if price <= 0:
            return
        if tickType == 4:
            self.last_price = price
            if self.callback:
                self.callback("price",price, datetime.now())
        elif tickType == 1:
            self.bid_price = price
        elif tickType == 2:
            self.ask_price = price
        

NameError: name 'EWrapper' is not defined

In [ ]:
class OHLCBar():
    def __init__(self, timestamp, open):
        self.timestamp = timestamp
        self.open = open 
        self.high = open
        self.low = open
        self.close = open
        self.tick_count = 1
        self.regime = 0 # Vol. regime (0: low, 1: med, 2: high)


    def update(self, price):
        self.high = max(self.high, price)
        self.low = min(self.low, price)
        self.close = price
        self.tick_count += 1 

    @property 
    def volatility(self): # Replace with Yang-Zhang from vol_service.py
        return (self.high - self.low) / self.close if self.close > 0 else 0


In [ ]:
# Replace with regime_service.py 
class MarkovRegime: 
    def ___init__(self):
        pass
    def calibrate(self,hist_bars):
        pass
    def _gaussian_likelihood(self, vol, regime):
        pass
    def get_regime(self, bars):
        pass

In [ ]:
class MarketDashboardMockup:
    def __init__(self, root): 
        self.root = root
        self.root.title("Market Dashboard")
        self.root.geometry("1200x800")

        self.IBApp = IBApp(callback=self.on_tick_data)
        self.connected = False
        self.streaming = False

        self.bar_duration = 5 # 5 seconds
        self.max_bars = 10
        self.ohlc_bars = deque(maxlen=self.max_bars)
        self.curr_bar = None
        self.bar_start = None
        self.price_history = deque(maxlen=100)

        self.setup_ui()
        self.setup_chart()
        self.setup_controls()

    
    # Skip all UI
    # Establish connection to IB

    def create_contract(self, symbol):
        # Input a Symbol
        contract = Contract()
        contract.symbol = symbol.upper()
        contract.secType = "STK"
        contract.exchange = "SMART"
        contract.currency = "USD"
        return contract 


    def connect_to_ib(self):
        try:
            host = self.host_var.get() 
            port = self.port_var.get()

            def connect_thread():
                try:
                    self.IBApp.connect(host, port, clientId=1)
                    self.IBApp.run()
                except Exception as e:
                    print(f"Error connecting to IB: {e}")
            thread = threading.Thread(target=connect_thread, daemon =True)
            thread.start()
            for i in range(50):
                if self.IBApp.connected:
                    break
                time.sleep(0.1)
            if not self.IBApp.connected:
                # TODO: Show error message in UI
                # TODO: Use logger to log errors
                # TODO: Handle connection errors gracefully
                raise Exception("Failed to connect to IB")
            print("Connected to IB")
            self.connected = True 
            # TODO: Disable "Connect" functionality in the UI
        except Exception as e:
            # TODO: Show error message in UI and log error
            print(f"Error connecting to IB: {e}")

        def disconnect_ib(self):
            try: 
                if self.streaming:
                    self.stop_stream()
                self.IBApp.disconnect()
                self.connected = False
            except Exception as e:
                print(f"Disconnect from IB error: {e}")
        # TODO: Enable Disconnect functionality in the UI
        # Such as disable disconnect and enable connect buttons

        def toggle_streaming(self):
            if not self.streaming:
                self.start_streaming()
            else:
                self.stop_streaming()
        # TODO: Enable Toggle Streaming functionality in the UI
        def start_streaming(self):
            if not self.connected:
                return
            symbol = self.symbol_var.get().upper()
            if not symbol:
                # TODO: Show error message in UI and make user enter a symbol  
                # TODO: Use logger to log errors
                return
            with self.bars_lock:
                self.ohlc_bars.clear()
                self.curr_bar = None
                self.bar_start = None
                self.price_history.clear() 
                self.regime = MarkovRegime() # Reinitialize MarkovRegime model / clear it
            
            contract = self.create_contract(symbol) 
            self.IBApp.historical_data.clear()
            self.IBApp.histDone.clear() 
            self.IBApp.reqHistoricalData(2, contract, "", "300 S", "5 secs", "TRADES", 1, 1, False, [])

            if self.IBApp.histDone.wait(timeout=10) and 2 in self.IBApp.historical_data:
                self.regime.calibrate(self.IBApp.historical_data[2])
                print(f"Regime model calibrated with {len(self.IBApp.historical_data[2])} bars")

            self.IBApp.reqMktData(1, contract, "", False, False, [])
            
            self.streaming = True
            self.running = True
            # TODO: Enable Streaming functionality in the UI
            # TODO: Update functionality to stop streaning on a button click 

            self.update_thread = threading.Thread(target=self.bar_manager_loop, daemon=True)
            self.update_thread.start()
            self.update_chart_loop()

        def stop_streaming(self):
            self.streaming = False
            self.running = False 
            try:
                self.IBApp.cancelMktData(1)
            except Exception as e:
                print(f"Error cancelling market data: {e}")

            # TODO: Update states and functionality

        def recalibrate_model(self):
            pass
            
        def on_tick_data(self, data_type, price, timestamp):
            if data_type == "price" and price > 0: 
                with self.bars_lock:
                    self.price_history.append((timestamp, price)) 

                    if self.curr_bar is None:
                        self.curr_bar = OHLCBar(timestamp, price)
                        self.bar_start = timestamp 
                    else:
                        self.curr_bar.update(price) 

                self.root.after(0, lambda: self.price_label.config(text=f"Current Price: ${price:.2f}"))
        
        def bar_manager_loop(self):
            while self.running:
                time.sleep(.1)
                with self.bars_lock:
                    if self.curr_bar is not None and self.bar_start is not None:
                        elapsed = (datetime.now() - self.bar_start).total_seconds()

                        if elapsed >= self.bar_duration:
                            self.ohlc_bars.append(self.curr_bar)
                            self.regime.get_regime(list(self.ohlc_bars))
                            last_price = self.curr_bar.close
                            self.curr_bar = OHLCBar(datetime.now(), last_price)
                            self.bar_start = datetime.now()

        def update_chart_loop(self):
            if not self.running:
                return
            self.draw_ohlc_chart()
            self.update_stats()
            self._after_id = self.root.after(200, self.update_chart_loop)


        def draw_ohlc_chart(self):
            # TODO: Implement UI parts

            pass
        
        def update_stats(self):
            while self.bar_lock:
                bars = list(self.ohlc_bars)
                current = self.curr_bar

            if current:
                bars = bars + [current]
            
            if not bars:
                return
            
            self.stats_labels["Bars"].config(text=f"Bars: {len(bars)}")

            all_highs = [b.high for b in bars]
            all_lows = [b.low for b in bars]
            self.stats_labels["High"].config(text=f"High: {max(all_highs):.2f}")
            self.stats_labels["Low"].config(text=f"Low: {min(all_lows):.2f}")
            
            curr_regime = bars[-1].regime if bars else 0
            self.stats_labels["Regime"].config(text=f"Regime: {curr_regime}")

            if current: 
                self.stats_labels["Ticks/Bar"].config(text=f"Ticks/Bar: {current.tick_count}")

        
        def on_closing(self):
            self.running = False
            if hasattr(self, "_after_id"):
                self.root.after_cancel(self._after_id)
            
            if self.connected:
                try:
                    if self.streaming:
                        self.IBApp.cancelMktData(1)
                    self.IBApp.disconnect()
                except Exception as e:
                    print(f"Error disconnecting from IB: {e}")

            self.root.destroy()

In [ ]:
def main():
    root =tk.Tk()
    app = MarketDashboardMockup(root)
    root.protocol("WM_DELETE_WINDOW", app.on_closing)
    root.mainloop()

if __name__ == "__main__":
    